In [ ]:
import dspy
import ipywidgets as widgets
from IPython.display import display, HTML
import boto3
import json
import os
import faiss
import io  
import numpy as np  
from typing import List, Dict
from datetime import datetime
import pandas as pd  # For interactive results table

# Initialize S3 and Bedrock clients
s3 = boto3.client('s3')
bedrock = boto3.client('bedrock-runtime')

# Configuration Dictionary
CONFIG = {
    "bucket_name": "sagemaker-us-east-1-188494237500",
    "json_folder": "dev/json",
    "index_folder": "dev/idx",
    "nlist": 512,
    "nprobe": 16,
    "num_training_samples": 1000,
    "embedding_models": {
        "amazon.titan-embed-text-v2": 1536,
        "openai.text-embedding-ada-002": 768,
        "cohere.embed-multilingual-v3": 1024
    }
}

# Dropdown for selecting actions
action_selector = widgets.Dropdown(
    options=[
        ("Generate Embeddings", "generate"),
        ("Run FAISS Search", "search")
    ],
    description="Action:",
    style={'description_width': 'initial'}
)

execute_button = widgets.Button(description="Execute")

# Live Logging Widget
log_output = widgets.Output(layout={'border': '1px solid black', 'width': '90%', 'height': '200px', 'overflow_y': 'auto'})

def log_message(message):
    """Updates the log output with messages."""
    with log_output:
        print(message)

# Display section header
def display_header(title):
    """Displays a visually enhanced section header."""
    display(HTML(f"<h2 style='color: navy; background-color: lightgray; padding: 10px; border-radius: 8px;'>{title}</h2>"))

# Query Cache
query_cache = {}  # Stores query embeddings per model
last_updated = datetime.now()  # Tracks when embeddings were last processed

# Bedrock Embedding Function with Cache
def generate_embedding(text, model_name):
    """Generate embedding using a specific Bedrock model, checking cache first."""
    cache_key = (text, model_name)

    if cache_key in query_cache:
        log_message(f"⚡ Using cached embedding for '{text}' ({model_name})")
        return query_cache[cache_key]

    response = bedrock.invoke_model(
        modelId=model_name,
        contentType="application/json",
        accept="application/json",
        body=json.dumps({"inputText": text})
    )
    response_body = json.loads(response['Body'].read().decode('utf-8'))
    embedding = response_body.get('embedding', [])

    expected_dim = CONFIG["embedding_models"].get(model_name, None)
    if expected_dim and len(embedding) != expected_dim:
        log_message(f"⚠️ Embedding dimension mismatch for {model_name}: Expected {expected_dim}, got {len(embedding)}")
        return []

    query_cache[cache_key] = embedding
    return embedding

# Load FAISS Index
def load_faiss_index_s3(config: Dict, model_name: str) -> faiss.Index:
    """Loads a FAISS index for a specific embedding model."""
    embedding_dim = config["embedding_models"].get(model_name, 1536)
    index_filename = f"faiss_index_{model_name}_{embedding_dim}.faiss"
    index_path = f"/tmp/{index_filename}"

    try:
        s3.download_file(config["bucket_name"], f"{config['index_folder']}/{index_filename}", index_path)
        index = faiss.read_index(index_path)
        log_message(f"✅ FAISS index loaded for {model_name} ({embedding_dim} dimensions).")
        return index
    except Exception as e:
        log_message(f"⚠️ No existing FAISS index for {model_name}. Creating a new one.")
        return None

# Save FAISS Index
def save_faiss_index_s3(config: Dict, index: faiss.Index, model_name: str) -> None:
    """Saves a FAISS index for a specific model."""
    embedding_dim = config["embedding_models"].get(model_name, 1536)
    index_filename = f"faiss_index_{model_name}_{embedding_dim}.faiss"
    index_path = f"/tmp/{index_filename}"

    try:
        faiss.write_index(index, index_path)
        s3.upload_file(index_path, config["bucket_name"], f"{config['index_folder']}/{index_filename}")
        log_message(f"✅ FAISS index saved for {model_name} ({embedding_dim} dimensions).")
    except Exception as e:
        log_message(f"❌ Error saving FAISS index for {model_name}: {e}")

# Process Embeddings and Refresh Cache
def process_embeddings():
    """Processes JSON files and builds FAISS index for each model separately."""
    global last_updated, query_cache  

    embedding_models = CONFIG["embedding_models"].keys()
    indices = {}

    log_message("🚀 Initializing FAISS indexes for models...")

    for model_name in embedding_models:
        index = load_faiss_index_s3(CONFIG, model_name)

        if index is None:
            embedding_dim = CONFIG["embedding_models"].get(model_name, 1536)
            quantizer = faiss.IndexFlatL2(embedding_dim)
            index = faiss.IndexIVFFlat(quantizer, embedding_dim, CONFIG["nlist"], faiss.METRIC_L2)

        indices[model_name] = index

    json_keys = list_objects_s3(CONFIG["bucket_name"], CONFIG["json_folder"])
    for json_key in json_keys:
        response = s3.get_object(Bucket=CONFIG["bucket_name"], Key=json_key)
        json_data = json.loads(response['Body'].read().decode('utf-8'))

        if "embeddings" not in json_data:
            continue

        for model_name, model_data in json_data["embeddings"].items():
            if "embedding" in model_data:
                embedding_array = np.array(model_data["embedding"], dtype=np.float32).reshape(1, CONFIG["embedding_models"][model_name])
                indices[model_name].add(embedding_array)

    for model_name, index in indices.items():
        save_faiss_index_s3(CONFIG, index, model_name)

    last_updated = datetime.now()
    query_cache.clear()
    log_message(f"✅ Embedding data refreshed at {last_updated}.")

# FAISS Search with Cache Check
def search_faiss_indexes_with_text(config: Dict, query: str, top_k: int = 5):
    """Searches FAISS indexes and returns results in an interactive table, checking cache age."""
    
    global query_cache, last_updated

    embedding_models = CONFIG["embedding_models"].keys()
    results_list = []

    log_message(f"🔍 Searching for: '{query}'")

    if any(time > last_updated for _, time in query_cache.values()):
        log_message("♻️ Clearing outdated query cache...")
        query_cache.clear()

    for model_name in embedding_models:
        try:
            index = load_faiss_index_s3(config, model_name)
            if index is None:
                log_message(f"⚠️ Skipping {model_name}, FAISS index not found.")
                continue

            query_embedding = generate_embedding(query, model_name)
            query_embedding = np.array(query_embedding, dtype=np.float32).reshape(1, CONFIG["embedding_models"][model_name])

            distances, indices = index.search(query_embedding, top_k)

            for rank, idx in enumerate(indices[0]):
                results_list.append({
                    "Model": model_name,
                    "Rank": rank + 1,
                    "Index Position": idx,
                    "Distance": distances[0][rank]
                })

        except Exception as e:
            log_message(f"❌ Error searching {model_name}: {e}")

    if results_list:
        df = pd.DataFrame(results_list)
        import ace_tools as tools
        tools.display_dataframe_to_user(name="Search Results", dataframe=df)

def execute_action(b):
    if action_selector.value == "generate":
        process_embeddings()
    elif action_selector.value == "search":
        query = input("Enter search query: ")
        search_faiss_indexes_with_text(CONFIG, query, top_k=5)

execute_button.on_click(execute_action)

display_header("🔹 Manage Embeddings & Search")
display(action_selector, execute_button, log_output)
